# Data Processing FEUX_PROV.gpkg
Here is the data processing I used for my first dataset

In [6]:
import geopandas as gpd
import numpy as np
import json

# Specify layer, since there is also a meta data layer in the geopackage
file = gpd.read_file("../site/data/FEUX_PROV.gpkg", layer="FEUX_PROV")
meta_data = gpd.read_file("../site/data/FEUX_PROV.gpkg", layer="meta_feux_prov")

## Projecting the data
I will be using MapLibre to display my map rather than leaflet, since I am dealing with a huge dataset. MapLibre Natively uses ESPG: 3857 for tile coordinates. 

In [7]:
# Checking the projected coordinate system and other info
print(file.crs)

EPSG:32198


**ESPG: 32198** <br>
NAD83 / Quebec Lambert


In [8]:
# Reprojecting into web mercator for web mapping
file = file.to_crs(epsg=3857)  

#Checking the geometry values to see if they make sense
print(file.head())

# Checking the crs now 
print(file.crs)


              geoc_fmj exercice origine an_origine perturb an_perturb  \
0  -89084,02+575646,54     2018    None       None     BRP       2018   
1  -90758,88+575699,39     2018    None       None     BRP       2018   
2  -91624,88+575810,41     2018    None       None     BRP       2018   
3  -88571,25+575953,70     2018    None       None     BRP       2018   
4  -89361,88+575940,30     2018    None       None     BRP       2018   

  part_str met_at_str  superficie  shape_length   shape_area  \
0       B1    TRIMSAT         0.6    469.296589   6209.95085   
1       B1    TRIMSAT         0.1    317.636672   1313.12070   
2       B1    TRIMSAT         1.3   1005.386315  12522.95030   
3       B1    TRIMSAT         0.8   1284.381280   8231.89760   
4       B1    TRIMSAT         3.0   2231.155437  29553.33580   

                                            geometry  
0  MULTIPOLYGON (((-7762212.043 6304884.032, -776...  
1  MULTIPOLYGON (((-7764713.818 6304825.101, -776...  
2  MULTIPOL

In [9]:
# Saving the new projected data file (this only includes the FEUX_PROV layer, not the meta data layer)
output3857 = "../site/data/FEUX_PROV_3857.gpkg"
file.to_file(output3857, driver="GPKG")

file = gpd.read_file("../site/data/FEUX_PROV_3857.gpkg")

## Getting to know the data
Checking the number of rows, which rows have no year of original disturbance, and what CRS the dataset is in. <br> <br>
Currently deciding if I should get rid of rows without years or keep them... Not sure

In [11]:
print(file.columns.tolist())

print("\nNumber of rows: ", len(file))

# If there are any rows without a year
no_exercice = file[file['exercice'].isna()]
print(f"Number of records with no exercise: {len(no_exercice)}")
# If there are any rows without geometry projected on to the map
no_geometry  = file[file['geometry'].isna()]
print('Number of records with no geometry: ', len(no_geometry))

['geoc_fmj', 'exercice', 'origine', 'an_origine', 'perturb', 'an_perturb', 'part_str', 'met_at_str', 'superficie', 'shape_length', 'shape_area', 'geometry']

Number of rows:  94572
Number of records with no exercise: 0
Number of records with no geometry:  0


**Number of rows total:** 94561
<br>**Number of rows without a original year:** 16656
<br>**Number of rows without geometry:** 0

In [12]:
print(file.head())

              geoc_fmj exercice origine an_origine perturb an_perturb  \
0  -89084,02+575646,54     2018    None       None     BRP       2018   
1  -90758,88+575699,39     2018    None       None     BRP       2018   
2  -91624,88+575810,41     2018    None       None     BRP       2018   
3  -88571,25+575953,70     2018    None       None     BRP       2018   
4  -89361,88+575940,30     2018    None       None     BRP       2018   

  part_str met_at_str  superficie  shape_length   shape_area  \
0       B1    TRIMSAT         0.6    469.296589   6209.95085   
1       B1    TRIMSAT         0.1    317.636672   1313.12070   
2       B1    TRIMSAT         1.3   1005.386315  12522.95030   
3       B1    TRIMSAT         0.8   1284.381280   8231.89760   
4       B1    TRIMSAT         3.0   2231.155437  29553.33580   

                                            geometry  
0  MULTIPOLYGON (((-7762212.043 6304884.032, -776...  
1  MULTIPOLYGON (((-7764713.818 6304825.101, -776...  
2  MULTIPOL

## Getting rid of unnescarry columns and simplifying the geometry 
To make the file load smoother, and be less heavy, I am getting rid of the columns I won't be using, which will get rid of about half the size of the file. I will also be simplify the geomtry to a 0.01m tolerance to further simplify the geomtry. 

In [29]:
file_filtered = file.drop(
    columns=['geoc_fmj', 'perturb', 'an_perturb', 'shape_area', 'shape_length'])

file_filtered.simplify(0.01)
file_filtered.to_file("../site/data/filtered_fuex.gpkg")

# All Checks Passed
Now that I have fully parsed through all my data and made all neseccary changes, I am will put it through the all checks passed from Lab3/4 to make sure there are no mistakes. 

In [5]:
import geopandas as gpd
import os

# Settings - Changed for my own data
FILE = "../site/data/filtered_fuex.gpkg"
WEST, EAST, SOUTH, NORTH = -8888861.34, -6356342.92, 5619947.33, 9007674.59   # a box around your area, in degrees NOTE: Using web mercator so mine is in meters
EXPECTED = 94572          # how many features you expect (rows after your filters): NOTE: didn't drop any rows 
POPUP_FIELDS = ["an_origine", "exercice", "superficie"]

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
# if back.crs is None or back.crs.to_epsg() != 4326:
if back.crs is None or back.crs.to_epsg() != 3857: # Web Mercator is necessary for tiles
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:3857).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")

# My file is HUGEEEEEEEEEEEEE that why it is turned into a postgresql database 
#mb = os.path.getsize(FILE) / 1_000_000
#if mb > 5:
    #problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['MultiPolygon']
All checks passed
